# **Laboratorio 8: Ready, Set, Deploy! 👩‍🚀👨‍🚀**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026 </strong></center>

### Cuerpo Docente:

- Profesores: Pablo Badilla, Diego Cortez
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Isadora Madrid

### **Link de repositorio de GitHub:** https://github.com/isadoramadrid/MDS7202.git

## Temas a tratar

- Entrenamiento y registro de modelos usando MLFlow.
- Despliegue de modelo usando FastAPI
- Containerización del proyecto usando Docker

### Objetivos principales del laboratorio

- Generar una solución a un problema a partir de ML
- Desplegar su solución usando MLFlow, FastAPI y Docker

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# **Introducción**

<p align="center">
  <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExODJnMHJzNzlkNmQweXoyY3ltbnZ2ZDlxY2c0aW5jcHNzeDNtOXBsdCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/AbPdhwsMgjMjax5reo/giphy.gif" width="400">
</p>



Consumida en la tristeza el despido de Renacín, Smapina ha decaído en su desempeño, lo que se ha traducido en un irregular tratamiento del agua. Esto ha implicado una baja en la calidad del agua, llegando a haber algunos puntos de la comuna en la que el vital elemento no es apto para el consumo humano. Es por esto que la sanitaria pública de la municipalidad de Maipú se ha contactado con ustedes para que le entreguen una urgente solución a este problema (a la vez que dejan a Smapina, al igual que Renacín, sin trabajo 😔).

El problema que la empresa le ha solicitado resolver es el de elaborar un sistema que les permita saber si el agua es potable o no. Para esto, la sanitaria les ha proveido una base de datos con la lectura de múltiples sensores IOT colocados en diversas cañerías, conductos y estanques. Estos sensores señalan nueve tipos de mediciones químicas y más una etiqueta elaborada en laboratorio que indica si el agua es potable o no el agua.

La idea final es que puedan, en el caso que el agua no sea potable, dar un aviso inmediato para corregir el problema. Tenga en cuenta que parte del equipo docente vive en Maipú y su intoxicación podría implicar graves problemas para el cierre del curso.

Atributos:

1. pH value
2. Hardness
3. Solids (Total dissolved solids - TDS)
4. Chloramines
5. Sulfate
6. Conductivity
7. Organic_carbon
8. Trihalomethanes
9. Turbidity

Variable a predecir:

10. Potability (1 si es potable, 0 no potable)

Descripción de cada atributo se pueden encontrar en el siguiente link: [dataset](https://www.kaggle.com/adityakadiwal/water-potability)

# **1. Optimización de modelos con Optuna + MLFlow (2.0 puntos)**

El objetivo de esta sección es que ustedes puedan combinar Optuna con MLFlow para poder realizar la optimización de los hiperparámetros de sus modelos.

Como aún no hemos hablado nada sobre `MLFlow` cabe preguntarse: **¡¿Qué !"#@ es `MLflow`?!**

<p align="center">
  <img src="https://media.tenor.com/eusgDKT4smQAAAAC/matthew-perry-chandler-bing.gif" width="400">
</p>

## **MLFlow**

`MLflow` es una plataforma de código abierto que simplifica la gestión y seguimiento de proyectos de aprendizaje automático. Con sus herramientas, los desarrolladores pueden organizar, rastrear y comparar experimentos, además de registrar modelos y controlar versiones.

<p align="center">
  <img src="https://spark.apache.org/images/mlflow-logo.png" width="350">
</p>

Si bien esta plataforma cuenta con un gran número de herramientas y funcionalidades, en este laboratorio trabajaremos con dos:
1. **Runs**: Registro que constituye la información guardada tras la ejecución de un entrenamiento. Cada `run` tiene su propio run_id, el cual sirve como identificador para el entrenamiento en sí mismo. Dentro de cada `run` podremos acceder a información como los hiperparámetros utilizados, las métricas obtenidas, las librerías requeridas y hasta nos permite descargar el modelo entrenado.
2. **Experiments**: Se utilizan para agrupar y organizar diferentes ejecuciones de modelos (`runs`). En ese sentido, un experimento puede agrupar 1 o más `runs`. De esta manera, es posible también registrar métricas, parámetros y archivos (artefactos) asociados a cada experimento.

### **Todo bien pero entonces, ¿cómo se usa en la práctica `MLflow`?**

Es sencillo! Considerando un problema de machine learning genérico, podemos registrar la información relevante del entrenamiento ejecutando `mlflow.autolog()` antes entrenar nuestro modelo. Veamos este bonito ejemplo facilitado por los mismos creadores de `MLflow`:

```python
#!pip install mlflow
import mlflow # importar mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog() # registrar automáticamente información del entrenamiento
with mlflow.start_run(): # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train) # train the model
    predictions = rf.predict(X_test) # Use the model to make predictions on the test dataset.
    # aquí termina el run
```

Si ustedes ejecutan el código anterior en sus máquinas locales (desde un jupyter notebook por ejemplo) se darán cuenta que en su directorio *root* se ha creado la carpeta `mlruns`. Esta carpeta lleva el tracking de todos los entrenamientos ejecutados desde el directorio root (importante: si se cambian de directorio y vuelven a ejecutar el código anterior, se creará otra carpeta y no tendrán acceso al entrenamiento anterior). Para visualizar estos entrenamientos, `MLflow` nos facilita hermosa interfaz visual a la que podemos acceder ejecutando:

```
mlflow ui
```

y luego pinchando en la ruta http://127.0.0.1:5000 que nos retorna la terminal. Veamos en vivo algunas de sus funcionalidades!

<p align="center">
  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZXVuM3A5MW1heDFpa21qbGlwN2pyc2VoNnZsMmRzODZxdnluemo2bCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3o84sq21TxDH6PyYms/giphy.gif" width="400">
</p>

Les dejamos también algunos comandos útiles:

- `mlflow.create_experiment("nombre_experimento")`: Les permite crear un nuevo experimento para agrupar entrenamientos
- `mlflow.log_metric("nombre_métrica", métrica)`: Les permite registrar una métrica *custom* bajo el nombre de "nombre_métrica"


Si tiene problemas puede necesitar ejecutar `uv add "setuptools<82.0.0"`

In [1]:
import mlflow # importar mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog() # registrar automáticamente información del entrenamiento
with mlflow.start_run(): # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train) # train the model
    predictions = rf.predict(X_test) # Use the model to make predictions on the test dataset.
    # aquí termina el run

2026/06/07 23:21:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [2]:
run = mlflow.last_active_run()
info = mlflow.get_run(run.info.run_id)
print(info.data.params)
print(info.data.metrics)

{'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_depth': '6', 'max_features': '3', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '1', 'min_samples_split': '2', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '100', 'n_jobs': 'None', 'oob_score': 'False', 'random_state': 'None', 'verbose': '0', 'warm_start': 'False'}
{'training_mean_absolute_error': 31.120933080977636, 'training_mean_squared_error': 1387.684781011596, 'training_r2_score': 0.7641798414304986, 'training_root_mean_squared_error': 37.25164131970021, 'training_score': 0.7641798414304986}


## **1.1 Combinando Optuna + MLflow (2.0 puntos)**

Ahora que tenemos conocimiento de ambas herramientas, intentemos ahora combinarlas para **más sabor**. El objetivo de este apartado es simple: automatizar la optimización de los parámetros de nuestros modelos usando `Optuna` y registrando de forma automática cada resultado en `MLFlow`.

Considerando el objetivo planteado, se le pide completar la función `optimize_model`, la cual debe:
- **Optimizar los hiperparámetros del modelo `XGBoost` usando `Optuna`.** Realice una cantidad de iteraciones para evitar tiempos de ejecución excesivos (al menos 10)
- **Registrar cada entrenamiento en un experimento nuevo**, asegurándose de que la métrica `f1-score` se registre como `"valid_f1"`. No se deben guardar todos los experimentos en *Default*; en su lugar, cada `experiment` y `run` deben tener nombres interpretables, reconocibles y diferentes a los nombres por defecto (por ejemplo, para un run: "XGBoost con lr 0.1").
- **Devolver el mejor modelo** usando la función `get_best_model` y serializarlo en el disco con `pickle.dump`. Luego, guardar el modelo en la carpeta `/models`.
- **Guardar el código en `optimize.py`**. La ejecución de `python optimize.py` debería ejecutar la función `optimize_model`.
- **Guardar las versiones de las librerías utilizadas** en el desarrollo.

*Hint: Le puede ser útil revisar los parámetros que recibe `mlflow.start_run`*

```python
def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_id)
    best_model_id = runs.sort_values("metrics.valid_f1")["run_id"].iloc[0]
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")

    return best_model
```

In [3]:
import os
import pickle
import pandas as pd
import optuna
import mlflow
import mlflow.sklearn
import xgboost

from xgboost import XGBClassifier
from sklearn.metrics import f1_score

c:\Users\Acer\OneDrive\Escritorio\MDS7202\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/06/07 23:22:37 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.


In [5]:
def get_best_model(experiment_id):
    # Busco todos los runs que quedaron guardados dentro del experimento de MLflow.
    runs = mlflow.search_runs(experiment_id)

    # Ordeno los runs según la métrica valid_f1, de mayor a menor,
    # porque quiero quedarme con el modelo que tuvo mejor desempeño.
    best_model_id = runs.sort_values("metrics.valid_f1", ascending=False)["run_id"].iloc[0]

    # Cargo desde MLflow el modelo asociado al mejor run.
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")
    
    return best_model


def optimize_model():
    # Primero cargo la base de datos del problema de potabilidad del agua.
    df = pd.read_csv("water_potability.csv")

    # Separo las variables predictoras X de la variable que quiero predecir, Potability.
    X = df.drop(columns=["Potability"])
    y = df["Potability"]

    # Divido los datos en entrenamiento y validación.
    # Uso stratify para mantener una proporción similar de clases 0 y 1 en ambos conjuntos.
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Creo o selecciono un experimento nuevo en MLflow para no guardar los resultados en Default.
    experiment_name = "XGBoost Optuna Water Potability"
    mlflow.set_experiment(experiment_name)

    # Obtengo el ID del experimento, porque lo necesito para registrar los runs correctamente.
    experiment = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id

    # Defino la función objetivo que Optuna va a intentar maximizar.
    # Cada trial corresponde a un entrenamiento con una combinación distinta de hiperparámetros.
    def objective(trial):

        # Aquí le digo a Optuna qué hiperparámetros quiero probar y en qué rangos.
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "random_state": 42,
            "eval_metric": "logloss"
        }

        # Le doy un nombre interpretable al run para poder identificarlo fácilmente en MLflow.
        run_name = f"XGBoost trial {trial.number}"

        # Abro un run en MLflow para registrar este entrenamiento específico.
        with mlflow.start_run(experiment_id=experiment_id, run_name=run_name):

            # Creo el modelo XGBoost con los hiperparámetros propuestos por Optuna.
            model = XGBClassifier(**params)

            # Entreno el modelo usando los datos de entrenamiento.
            model.fit(X_train, y_train)

            # Predigo sobre el conjunto de validación.
            y_pred = model.predict(X_valid)

            # Calculo el f1-score, que será la métrica que voy a optimizar.
            valid_f1 = f1_score(y_valid, y_pred)

            # Registro en MLflow los hiperparámetros usados en este entrenamiento.
            mlflow.log_params(params)

            # Registro la métrica principal con el nombre que pide el enunciado: valid_f1.
            mlflow.log_metric("valid_f1", valid_f1)

            # Guardo el modelo entrenado dentro del run de MLflow.
            mlflow.sklearn.log_model(model, "model")

        # Retorno el f1-score para que Optuna sepa qué tan bueno fue este trial.
        return valid_f1

    # Creo el estudio de Optuna indicando que quiero maximizar la métrica valid_f1.
    study = optuna.create_study(direction="maximize")

    # Ejecuto 10 iteraciones de búsqueda de hiperparámetros, como mínimo pedido por el enunciado.
    study.optimize(objective, n_trials=10)

    # Una vez terminada la optimización, recupero desde MLflow el mejor modelo encontrado.
    best_model = get_best_model(experiment_id)

    # Creo la carpeta models si todavía no existe.
    os.makedirs("models", exist_ok=True)

    # Guardo el mejor modelo en disco usando pickle.
    with open("models/best_xgboost_model.pkl", "wb") as file:
        pickle.dump(best_model, file)

    # Guardo las versiones de las principales librerías utilizadas.
    # Esto sirve para dejar registro del ambiente usado en el laboratorio.
    with open("models/requirements.txt", "w") as file:
        file.write(f"pandas=={pd.__version__}\n")
        file.write(f"optuna=={optuna.__version__}\n")
        file.write(f"mlflow=={mlflow.__version__}\n")
        file.write(f"xgboost=={xgboost.__version__}\n")

    # Finalmente retorno el mejor modelo, por si quiero usarlo después en el notebook.
    return best_model

In [6]:
best_model = optimize_model()

[I 2026-06-07 23:30:58,197] A new study created in memory with name: no-name-210bbd88-ec7d-4901-8337-ae314fd7f25a
2026/06/07 23:31:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 23:31:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Acer\OneDrive\Escritorio\MDS7202\.venv\lib\site-packages\xgboost\core.py:158: UserWarning: [23:31:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats."
2026/06/07 23:31:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 23:31:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer t

# **2. FastAPI (2.0 puntos)**

<div align="center">
  <img src="https://media3.giphy.com/media/YQitE4YNQNahy/giphy-downsized-large.gif" width="500">
</div>

Con el modelo ya entrenado, la idea de esta sección es generar una API REST a la cual se le pueda hacer *requests* para así interactuar con su modelo. En particular, se le pide:

- Guardar el código de esta sección en el archivo `main.py`. Note que ejecutar `python main.py` debería levantar el servidor en el puerto por defecto.
- Defina `GET` con ruta tipo *home* que describa brevemente su modelo, el problema que intenta resolver, su entrada y salida.
- Defina un `POST` a la ruta `/potabilidad/` donde utilice su mejor optimizado para predecir si una medición de agua es o no potable. Por ejemplo, una llamada de esta ruta con un *body*:

```json
{
   "ph":10.316400384553162,
   "Hardness":217.2668424334475,
   "Solids":10676.508475429378,
   "Chloramines":3.445514571005745,
   "Sulfate":397.7549459751925,
   "Conductivity":492.20647361771086,
   "Organic_carbon":12.812732207582542,
   "Trihalomethanes":72.28192021570328,
   "Turbidity":3.4073494284238364
}
```

Su servidor debería retornar una respuesta HTML con código 200 con:


```json
{
  "potabilidad": 0 # respuesta puede variar según el clasificador que entrenen
}
```

**`HINT:` Recuerde que puede utilizar [http://localhost:8000/docs](http://localhost:8000/docs) para hacer un `POST`.**

# **3. Docker (2 puntos)**

<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*9rafh2W0rbRJIKJzqYc8yA.gif" width="500">
</div>

Tras el éxito de su aplicación web para generar la salida, Smapina le solicita que genere un contenedor para poder ejecutarla en cualquier computador de la empresa de agua potable.

## **3.1 Creación de Container (1 punto)**

Cree un Dockerfile que use una imagen base de Python, copie los archivos del proyecto e instale las dependencias desde un `requirements.txt`. Con esto, construya y ejecute el contenedor Docker para la API configurada anteriormente. Entregue el código fuente (incluyendo `main.py`, `requirements.txt`, y `Dockerfile`) y la imagen Docker de la aplicación. Para la dockerización, asegúrese de cumplir con los siguientes puntos:

1. **Generar un archivo `.dockerignore`** que ignore carpetas y archivos innecesarios dentro del contenedor.
2. **Configurar un volumen** que permita la persistencia de los datos en una ruta local del computador.
3. **Exponer el puerto** para acceder a la ruta de la API sin tener que entrar al contenedor directamente.
4. **Incluir imágenes en el notebook** que muestren la ejecución del contenedor y los resultados obtenidos.
5. **Revisar y comentar los recursos utilizados por el contenedor**. Analice si los contenedores son livianos en términos de recursos.

## **3.2 Preguntas de Smapina (1 punto)**
Tras haber experimentado con Docker, Smapina desea profundizar más en el tema y decide realizarle las siguientes consultas:

- ¿Cómo se diferencia Docker de una máquina virtual (VM)?
> Docker utiliza contenedores que comparten el kernel del sistema operativo anfitrión, mientras que una máquina virtual ejecuta un sistema operativo completo sobre un hipervisor. Por eso, los contenedores suelen ser más livianos, rápidos de iniciar y consumir menos recursos que una VM.
- ¿Cuál es la diferencia entre usar Docker y ejecutar la aplicación directamente en el sistema local?
> Al ejecutar la aplicación directamente en el sistema local, dependo de que mi computador tenga instaladas las versiones correctas de Python, librerías y configuraciones necesarias. En cambio, con Docker todo queda definido dentro de una imagen: sistema base, dependencias, código y comando de ejecución. Esto reduce problemas de compatibilidad, ya que la aplicación se ejecuta dentro del mismo entorno sin importar el pc.

- ¿Cómo asegura Docker la consistencia entre diferentes entornos de desarrollo y producción?
> Docker asegura consistencia porque el entorno queda descrito en archivos como el Dockerfile y requirements.txt. Al construir la imagen, se instalan siempre las mismas dependencias y se ejecuta la aplicación de la misma forma. Así, si la imagen funciona en mi computador, debería funcionar de manera equivalente en otro entorno.

- ¿Cómo se gestionan los volúmenes en Docker para la persistencia de datos?
> Los volúmenes permiten conectar una carpeta del computador local con una carpeta dentro del contenedor. Esto es importante porque los datos creados dentro de un contenedor normalmente se pierden si el contenedor se elimina. En este laboratorio usé un volumen para conectar la carpeta local data con /app/data dentro del contenedor, permitiendo guardar las predicciones en predicciones.csv de forma persistente fuera del contenedor.

- ¿Qué son Dockerfile y docker-compose.yml, y cuál es su propósito?
>  El Dockerfile es el archivo que contiene las instrucciones para construir una imagen Docker. Por ejemplo, define la imagen base de Python, copia los archivos del proyecto, instala dependencias y especifica el comando para levantar la API.
> Por otro lado, docker-compose.yml sirve para definir y ejecutar uno o más contenedores de forma coordinada. Es útil cuando una aplicación necesita varios servicios, como una API, una base de datos y un servicio de monitoreo. 

## **Comentarios del lab**

> Se creó y ejecutó correctamente un contenedor Docker para la API de FastAPI. El contenedor expone el puerto 8000, permitiendo acceder a la documentación interactiva en http://localhost:8000/docs y realizar predicciones mediante el endpoint POST /potabilidad/.

> Además, se configuró un volumen entre la carpeta local data y la ruta /app/data dentro del contenedor. Esto permitió guardar de forma persistente las predicciones realizadas por la API en el archivo predicciones.csv, incluso fuera del contenedor.

> Al revisar los recursos con docker stats, el contenedor mostró un bajo consumo en ejecución: aproximadamente 0.21% de CPU y 121.5 MiB de memoria, equivalente a 1.55% del límite disponible. Esto indica que la API es liviana en términos de recursos mientras se encuentra en reposo o procesando pocas solicitudes. Sin embargo, la imagen Docker tiene un tamaño relativamente alto, cercano a 1.8GB de uso en disco, principalmente por las dependencias necesarias para ejecutar el modelo, como scikit-learn, pandas y xgboost.



### Evidencia de ejecución del contenedor

A continuación se muestra el contenedor ejecutándose correctamente con la API expuesta en el puerto 8000.

![Ejecución del contenedor](imagenes/imagen_1.png)

### Prueba del endpoint POST /potabilidad/

Se realizó una solicitud POST al endpoint `/potabilidad/`, obteniendo una respuesta con código 200 y la predicción de potabilidad.

![Respuesta del endpoint](imagenes/imagen_2.png)

### Recursos utilizados por el contenedor

Se revisó el uso de recursos con `docker stats --no-stream water-api`.

![Recursos Docker](imagenes/imagen_3.png)

# Conclusión

Éxito!
<div align="center">
  <img src="https://i.pinimg.com/originals/55/f5/fd/55f5fdc9455989f8caf7fca7f93bd96a.gif" width="500">
</div>